# Machine Learning - Assignment 2
## Hotel Booking Cancellation Prediction: comparison of six classification models

**M.Tech (AIML / DSE) - Work Integrated Learning Programmes Division**

| Field | Value |
|---|---|
| Name | SAIKAT SAHU |
| BITS ID | 2026CT05124 |
| Dataset | Hotel Booking Demand (Kaggle: `jessemostipak/hotel-booking-demand`) |
| Target | `is_canceled` (1 = booking cancelled, 0 = not cancelled) |

**How to run on Google Colab:** Runtime > Run all. When the data cell asks, upload `hotel_bookings.csv`
(or place it in the Colab *Files* sidebar first, and it will be found automatically). No other setup is needed.

**Notebook structure (follows the assignment steps):** 1. Dataset choice and description - 2. Preprocessing -
3. Models and evaluation metrics - 4. Comparison table and graphs - 5. Observations - 6. Files for the GitHub repository (README, requirements, test data).

In [ ]:
# Setup: install any missing package (Colab already has all of these, so this normally does nothing)
import importlib.util, subprocess, sys

required = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "seaborn": "seaborn",
            "sklearn": "scikit-learn", "xgboost": "xgboost"}
missing = [pip_name for mod, pip_name in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score, recall_score, f1_score,
                             matthews_corrcoef, confusion_matrix, roc_curve, ConfusionMatrixDisplay)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

SEED = 42                      # fixed random seed used everywhere
np.random.seed(SEED)
os.makedirs("outputs", exist_ok=True)
print("scikit-learn / pandas / numpy imported. XGBoost available:", HAS_XGB)

## Step 1 - Dataset choice

### 1.1 Loading the data
The cell below looks for `hotel_bookings.csv` in the working folder, `/content` and Google Drive.
If it is not found, Colab shows an upload button. If nothing is uploaded, it downloads the same public dataset
(Hotel Booking Demand, 119,390 rows) from a public GitHub mirror so the notebook never stops.

In [ ]:
import pandas as pd

GITHUB_USER = "Saikat2026ct05124"          # <- your GitHub username
REPO_NAME   = "ML_Assignment-2"                  # <- your repository name
BRANCH      = "main"                             # <- usually "main" (or "master")
FILE_PATH   = "data/hotel_booking_latest.csv"          # <- path of the CSV inside the repo

CSV_URL = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{FILE_PATH}"

try:
    df_raw = pd.read_csv(CSV_URL)
    print("Loaded from GitHub:", df_raw.shape)
except Exception as e:
    print("GitHub load failed:", e)
    from google.colab import files
    uploaded = files.upload()
    df_raw = pd.read_csv(next(iter(uploaded)))
    print("Loaded from upload:", df_raw.shape)

### 1.2 Why this dataset, target variable and problem statement
- **Source:** Kaggle - *Hotel Booking Demand* (Antonio, Almeida and Nunes, 2019): real bookings of a city hotel and a resort hotel in Portugal (2015-2017).
- **Problem statement:** predict whether a booking will be **cancelled** (`is_canceled`) using information known when the booking is made (lead time, deposit type, market segment, guest history and so on). Cancellations cost hotels revenue through empty rooms, so an early warning helps with overbooking and pricing decisions.
- **Why it fits the assignment:** 119,390 rows (min 1,000), 32 columns (min 12), real missing values (`children`, `country`, `agent`, `company`), many duplicate rows, a mix of numeric and categorical features, and a moderately imbalanced binary target.
- **Target variable:** `is_canceled` (binary: 0 = kept, 1 = cancelled).

In [ ]:
print("Number of instances :", df_raw.shape[0])
print("Number of features   :", df_raw.shape[1] - 1, "(excluding the target)")
print()

feature_types = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "n_unique": df_raw.nunique(),
    "n_missing": df_raw.isna().sum(),
    "pct_missing": (df_raw.isna().mean() * 100).round(2),
})
feature_types["type"] = np.where(df_raw.dtypes == "object", "categorical", "numeric")
feature_types

In [ ]:
# Class distribution of the target (class imbalance must be examined)
counts = df_raw["is_canceled"].value_counts().sort_index()
pct = (counts / counts.sum() * 100).round(2)
print(pd.DataFrame({"count": counts, "percent": pct}).rename(index={0: "0 - not cancelled", 1: "1 - cancelled"}))
print(f"\nImbalance ratio (majority : minority) = {counts.max() / counts.min():.2f} : 1")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(x="is_canceled", data=df_raw, ax=axes[0], palette=["#4c78a8", "#f58518"])
axes[0].set_title("Target class distribution")
axes[0].set_xticklabels(["Not cancelled (0)", "Cancelled (1)"])
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
miss = df_raw.isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)
miss.plot(kind="bar", ax=axes[1], color="#e45756")
axes[1].set_title("Missing values per column")
axes[1].set_ylabel("count")
plt.tight_layout()
plt.savefig("outputs/eda_target_missing.png", dpi=150)
plt.show()

In [ ]:
# Short EDA: how the strongest signals relate to cancellation
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.barplot(x="deposit_type", y="is_canceled", data=df_raw, ax=axes[0], palette="Set2")
axes[0].set_title("Cancellation rate by deposit type")
sns.barplot(x="hotel", y="is_canceled", data=df_raw, ax=axes[1], palette="Set2")
axes[1].set_title("Cancellation rate by hotel")
sns.boxplot(x="is_canceled", y="lead_time", data=df_raw, ax=axes[2], palette="Set2")
axes[2].set_title("Lead time vs cancellation")
plt.tight_layout()
plt.savefig("outputs/eda_relationships.png", dpi=150)
plt.show()

## Step 2 - Data preprocessing
Decisions made, in order:
1. **Duplicates** - exact duplicate rows are removed (before splitting, so identical bookings cannot leak between train and test).
2. **Target leakage** - `reservation_status` and `reservation_status_date` are dropped: they are only known *after* the outcome (a "Canceled" status is the target itself).
3. **Invalid rows** - bookings with zero guests (`adults + children + babies = 0`) are removed.
4. **Missing values** - `company` is dropped (~94% missing). `children`, `agent`, `country` are imputed **inside the pipeline**, so imputation values are learned from the training data only.
5. **Encoding** - one-hot encoding for categorical features (rare categories below 0.5% are grouped, unseen categories are ignored).
6. **Scaling** - `StandardScaler` for numeric features (needed by Logistic Regression, KNN; harmless for trees).
7. **Split** - 80:20, stratified (keeps the class ratio), `random_state = 42`.
8. **No leakage** - the preprocessing transformer is fitted on the training set only, then applied to the test set.

In [ ]:
df = df_raw.copy()
n_before = len(df)

# 1. duplicates
n_dup = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicate rows removed: {n_dup:,}  ({n_before:,} -> {len(df):,})")

# 2. target leakage columns
leak_cols = [c for c in ["reservation_status", "reservation_status_date"] if c in df.columns]
df = df.drop(columns=leak_cols)
print("Dropped leakage columns:", leak_cols)

# 3. bookings with no guests at all
zero_guests = (df["adults"].fillna(0) + df["children"].fillna(0) + df["babies"].fillna(0)) == 0
print(f"Rows with zero guests removed: {int(zero_guests.sum())}")
df = df[~zero_guests].reset_index(drop=True)

# 4. columns with an extreme share of missing values
high_missing = [c for c in df.columns if df[c].isna().mean() > 0.90]
df = df.drop(columns=high_missing)
print("Dropped columns with >90% missing:", high_missing)

# agent is an ID (category), not a quantity -> treat as categorical text
if "agent" in df.columns:
    df["agent"] = df["agent"].apply(lambda v: str(int(v)) if pd.notna(v) else np.nan)

print("\nShape after cleaning:", df.shape)
print("Remaining missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])

In [ ]:
TARGET = "is_canceled"
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Numeric features ({len(numeric_features)}):", numeric_features)
print(f"\nCategorical features ({len(categorical_features)}):", categorical_features)

# 7. stratified 80:20 split with a fixed seed
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
print(f"\nTrain: {X_train.shape}   Test: {X_test.shape}")
print("Class share in train :", y_train.value_counts(normalize=True).round(4).to_dict())
print("Class share in test  :", y_test.value_counts(normalize=True).round(4).to_dict())

In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.005, sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])

# Fit ONLY on training data, then transform both sets
X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

print("Processed train matrix:", X_train_p.shape)
print("Processed test matrix :", X_test_p.shape)
print("Any NaN left          :", bool(np.isnan(X_train_p).any() or np.isnan(X_test_p).any()))

## Step 3 - Classification models and evaluation metrics
The five models required by the assignment are implemented on the same preprocessed data:
**Logistic Regression, Decision Tree, K-Nearest Neighbors, Gaussian Naive Bayes, Random Forest (ensemble).**
The assignment text also refers to a comparison of *6* models, so **XGBoost** (boosting ensemble) is added as a sixth model.
If you only want the five listed models, set `INCLUDE_SIXTH_MODEL = False` in the next cell.

Metrics for every model: Accuracy, AUC, Precision, Recall, F1 and Matthews Correlation Coefficient (MCC).
The positive class is *cancelled* (1). AUC is computed from predicted probabilities.

In [ ]:
INCLUDE_SIXTH_MODEL = False

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, min_samples_leaf=5, random_state=SEED),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
    "Naive Bayes (Gaussian)": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED),
}
if INCLUDE_SIXTH_MODEL and HAS_XGB:
    models["XGBoost"] = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6, subsample=0.9,
                                      colsample_bytree=0.9, eval_metric="logloss", n_jobs=-1, random_state=SEED)
elif INCLUDE_SIXTH_MODEL:
    print("XGBoost is not installed - continuing with the five required models.")

print("Models to train:", list(models))

In [ ]:
results, fitted, predictions, probabilities, train_times = [], {}, {}, {}, {}

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train_p, y_train)
    train_times[name] = time.time() - t0

    y_pred = model.predict(X_test_p)
    y_prob = model.predict_proba(X_test_p)[:, 1]
    fitted[name], predictions[name], probabilities[name] = model, y_pred, y_prob

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_test, y_pred),
    })
    r = results[-1]
    print(f"{name:<24} acc={r['Accuracy']:.4f}  auc={r['AUC']:.4f}  prec={r['Precision']:.4f}  "
          f"rec={r['Recall']:.4f}  f1={r['F1']:.4f}  mcc={r['MCC']:.4f}   ({train_times[name]:.1f}s)")

results_df = pd.DataFrame(results).set_index("Model")
metric_cols = ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]

## Step 4 - Comparison table and graphs

In [ ]:
comparison = results_df[metric_cols].round(4)
comparison["Train time (s)"] = pd.Series(train_times).round(1)
comparison.to_csv("outputs/model_comparison.csv")
comparison.style.format("{:.4f}", subset=metric_cols).background_gradient(cmap="YlGn", subset=metric_cols)

In [ ]:
# Grouped bar chart: every metric for every model
ax = results_df[metric_cols].plot(kind="bar", figsize=(14, 5.5), width=0.82, colormap="tab10", edgecolor="white")
ax.set_title("Evaluation metrics by model")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.08)
ax.set_xticklabels(results_df.index, rotation=20, ha="right")
ax.legend(title="Metric", ncol=6, loc="upper center", bbox_to_anchor=(0.5, -0.22))
plt.tight_layout()
plt.savefig("outputs/metrics_bar_chart.png", dpi=150, bbox_inches="tight")
plt.show()

# One panel per metric (easier to read exact differences)
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, m in zip(axes.ravel(), metric_cols):
    vals = results_df[m].sort_values()
    ax.barh(vals.index, vals.values, color=sns.color_palette("viridis", len(vals)))
    ax.set_title(m)
    ax.set_xlim(min(0.0, vals.min()), min(1.0, vals.max() + 0.08) if m != "MCC" else 1.0)
    for i, v in enumerate(vals.values):
        ax.text(v, i, f" {v:.3f}", va="center")
plt.tight_layout()
plt.savefig("outputs/metrics_per_panel.png", dpi=150)
plt.show()

In [ ]:
# Heatmap of the table + ROC curves
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
sns.heatmap(results_df[metric_cols], annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[0], cbar=False)
axes[0].set_title("Metric heatmap")
axes[0].set_ylabel("")

for name in models:
    fpr, tpr, _ = roc_curve(y_test, probabilities[name])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={results_df.loc[name, 'AUC']:.3f})")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].set_title("ROC curves")
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.savefig("outputs/heatmap_roc.png", dpi=150)
plt.show()

In [ ]:
# Confusion matrices
n = len(models)
cols = 3
rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.4 * rows))
axes = np.atleast_1d(axes).ravel()
for ax, name in zip(axes, models):
    ConfusionMatrixDisplay(confusion_matrix(y_test, predictions[name]),
                           display_labels=["Not cancelled", "Cancelled"]).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(name)
for ax in axes[n:]:
    ax.axis("off")
plt.tight_layout()
plt.savefig("outputs/confusion_matrices.png", dpi=150)
plt.show()

In [ ]:
# Which features matter most? (Random Forest importance)
rf = fitted["Random Forest"]
imp = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False).head(15)
plt.figure(figsize=(8, 5))
sns.barplot(x=imp.values, y=[n.split("__", 1)[-1] for n in imp.index], palette="mako")
plt.title("Top 15 features - Random Forest importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("outputs/rf_feature_importance.png", dpi=150)
plt.show()
top_features = [n.split("__", 1)[-1] for n in imp.index[:5]]
print("Top 5 features:", top_features)

## Step 5 - Observations on model performance
The text below is generated from the actual numbers in the table above, so it always matches the results.
Read it, check it against the graphs, and **rewrite it in your own words** before you submit (the assignment checks originality).

In [ ]:
def rank_of(metric, name):
    return int(results_df[metric].rank(ascending=False, method="min")[name])


reasons = {
    "Logistic Regression": "it is a linear model: fast and stable on scaled/one-hot data, but it cannot capture feature interactions such as deposit type combined with lead time",
    "Decision Tree": "a single tree captures non-linear rules (for example non-refundable deposits) but is high-variance, so it generalises worse than ensembles",
    "K-Nearest Neighbors": "it relies on distances, which become less informative in a high-dimensional one-hot space, and it is slow at prediction time",
    "Naive Bayes (Gaussian)": "it assumes independent, Gaussian-distributed features, which is violated by one-hot columns and correlated booking features",
    "Random Forest": "averaging many de-correlated trees reduces variance and handles non-linearity and mixed feature types well",
    "XGBoost": "boosting corrects the errors of earlier trees step by step, usually giving the strongest tabular-data performance",
}

lines = []
best_mcc = results_df["MCC"].idxmax()
worst_mcc = results_df["MCC"].idxmin()
for name in results_df.index:
    r = results_df.loc[name]
    cm_ = confusion_matrix(y_test, predictions[name])
    tn, fp, fn, tp = cm_.ravel()
    lines.append(
        f"- **{name}**: accuracy {r['Accuracy']:.3f}, AUC {r['AUC']:.3f}, precision {r['Precision']:.3f}, recall {r['Recall']:.3f}, "
        f"F1 {r['F1']:.3f}, MCC {r['MCC']:.3f} (MCC rank {rank_of('MCC', name)} of {len(results_df)}). "
        f"It missed {fn:,} real cancellations (false negatives) and wrongly flagged {fp:,} bookings (false positives). "
        f"Likely reason: {reasons.get(name, 'see model description')}."
    )

summary_lines = [
    f"- Best overall model by MCC: **{best_mcc}** (MCC {results_df.loc[best_mcc, 'MCC']:.3f}, AUC {results_df.loc[best_mcc, 'AUC']:.3f}); weakest: **{worst_mcc}** (MCC {results_df.loc[worst_mcc, 'MCC']:.3f}).",
    f"- Highest recall (catches the most cancellations): **{results_df['Recall'].idxmax()}** ({results_df['Recall'].max():.3f}); highest precision: **{results_df['Precision'].idxmax()}** ({results_df['Precision'].max():.3f}).",
    f"- Class imbalance: {pct.get(1, 0):.1f}% of bookings are cancelled, so accuracy alone can look good while minority-class recall is poor. MCC, F1 and AUC give a fairer view.",
    f"- The most influential features in the Random Forest were: {', '.join(top_features)}.",
    f"- Ensemble methods (Random Forest{', XGBoost' if 'XGBoost' in results_df.index else ''}) vs the single models: "
    f"best ensemble MCC {results_df.loc[[m for m in ['Random Forest', 'XGBoost'] if m in results_df.index], 'MCC'].max():.3f} vs best single model MCC "
    f"{results_df.loc[[m for m in results_df.index if m not in ['Random Forest', 'XGBoost']], 'MCC'].max():.3f}.",
]

observations_md = "### Per-model observations\n" + "\n".join(lines) + "\n\n### Overall summary\n" + "\n".join(summary_lines)
from IPython.display import Markdown, display
display(Markdown(observations_md))

## Step 6 - Files for the GitHub repository
This cell writes the files the assignment asks for into the `outputs/` folder and prepares a download:
`README.md` (with the required structure and your results), `requirements.txt`, the test data CSV, and the comparison table CSV.
Then push them (plus this notebook) to your GitHub repository. Remember to fill in your repository link in the README and
add your BITS Virtual Lab screenshot to the PDF.

In [ ]:
# Test data used in the experiments (raw features + true label)
test_export = X_test.copy()
test_export[TARGET] = y_test.values
test_export.to_csv("outputs/test_data.csv", index=False)

requirements = "scikit-learn\nnumpy\npandas\nmatplotlib\nseaborn\nxgboost\n"
with open("outputs/requirements.txt", "w") as f:
    f.write(requirements)


def to_markdown_table(frame, floatfmt="{:.4f}"):
    cols = ["Model"] + list(frame.columns)
    out = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for idx, row in frame.iterrows():
        vals = [floatfmt.format(v) if isinstance(v, (float, np.floating)) else str(v) for v in row.values]
        out.append("| " + " | ".join([str(idx)] + vals) + " |")
    return "\n".join(out)


readme = f"""# Hotel Booking Cancellation Prediction - ML Assignment 2

**Name:** SAIKAT SAHU  |  **BITS ID:** 2026CT05124

## 1. Problem statement
Predict whether a hotel booking will be cancelled (`is_canceled`) from booking information, and compare six classification models
(Logistic Regression, Decision Tree, K-Nearest Neighbors, Gaussian Naive Bayes, Random Forest, XGBoost).

## 2. Dataset description
- Source: Kaggle - Hotel Booking Demand (https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand)
- Instances: {df_raw.shape[0]:,} raw rows ({len(df):,} after removing duplicates and invalid rows); features: {df_raw.shape[1] - 1} raw columns ({X.shape[1]} used after cleaning)
- Target: `is_canceled` (1 = cancelled). Class balance: {pct.get(0, 0):.1f}% not cancelled / {pct.get(1, 0):.1f}% cancelled
- Feature types: {len(numeric_features)} numeric, {len(categorical_features)} categorical
- Missing values: `children`, `country`, `agent`, `company` - `company` dropped (>90% missing), others imputed on training data only
- Preprocessing: duplicates removed, leakage columns (`reservation_status`, `reservation_status_date`) dropped, one-hot encoding, standard scaling,
  stratified 80:20 split with `random_state={SEED}`, transformers fitted on the training set only.

## 3. GitHub repository link
<PASTE YOUR REPOSITORY LINK HERE>

## 4. Models used and results
Model configurations: Logistic Regression (max_iter=1000), Decision Tree (max_depth=12, min_samples_leaf=5), KNN (k=7),
Gaussian Naive Bayes (defaults), Random Forest (200 trees), XGBoost (300 trees, depth 6, lr 0.1).

{to_markdown_table(comparison)}

Graphs: see `metrics_bar_chart.png`, `metrics_per_panel.png`, `heatmap_roc.png`, `confusion_matrices.png`.

## 5. Observations
{observations_md}

## 6. How to run
1. `pip install -r requirements.txt`
2. Place `hotel_bookings.csv` next to the notebook (or upload it in Colab when asked).
3. Run all cells of `ML_Assignment_2_Hotel_Cancellation.ipynb`.
"""
with open("outputs/README.md", "w") as f:
    f.write(readme)

print("Files written to ./outputs :")
for fn in sorted(os.listdir("outputs")):
    print(" -", fn, f"({os.path.getsize(os.path.join('outputs', fn)) / 1024:.0f} KB)")

In [ ]:
# Colab only: zip everything and download it in one click (skipped automatically elsewhere)
try:
    import shutil
    from google.colab import files
    shutil.make_archive("assignment_outputs", "zip", "outputs")
    files.download("assignment_outputs.zip")
except Exception as e:
    print("Download step skipped (not running in Colab):", type(e).__name__)